# 13 — Speech Recognition & Transcription — From Waveforms to Words

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/13_speech_recognition_and_transcription.ipynb)

⏱ **Reading time:** ~30 min &nbsp;|&nbsp; **Prerequisites:** Notebooks 1–12

Speech is arguably the most natural human interface. In this notebook we trace
the full journey from a raw audio waveform to a clean text transcript, using
OpenAI's **Whisper** family of models via Hugging Face Transformers.

## Learning Objectives

By the end of this notebook you will be able to:

* **Explain** the full ASR pipeline — waveform → spectrogram → features → tokens → text.
* **Visualize** waveforms, spectrograms, and mel-spectrograms and understand what each reveals.
* **Use Whisper** (via `transformers.pipeline`) for short- and long-form transcription.
* **Extract timestamps** at both segment and word level for subtitle generation.
* **Detect language** and translate speech across Whisper's 99 supported languages.
* **Evaluate** transcription quality with Word Error Rate (WER) and Character Error Rate (CER).

## 1 · How Speech Becomes Text

Every automatic speech recognition (ASR) system solves the same fundamental
problem: *given a continuous pressure wave captured by a microphone, produce a
sequence of discrete text tokens*. The pipeline looks roughly like this:

```
Analog sound  ──▶  ADC (16 kHz, 16-bit PCM)  ──▶  Digital waveform
     │                                                    │
     │                                   STFT / Mel filterbank
     │                                                    │
     ▼                                                    ▼
 Microphone                                    Mel-spectrogram
                                                          │
                                             Encoder (Transformer)
                                                          │
                                             Decoder (Transformer)
                                                          │
                                                    Text tokens
```

**Sampling rate** is the first critical parameter. Speech is conventionally
sampled at **16 kHz** — 16,000 amplitude measurements per second. By the
Nyquist theorem this captures frequencies up to 8 kHz, covering the
fundamental frequencies of human speech (85–300 Hz) plus formants and
fricatives that distinguish phonemes (up to ~7 kHz).

Once we have the raw waveform, we convert it to a **time-frequency
representation** — the spectrogram. A Short-Time Fourier Transform (STFT)
slides a window (typically 25 ms, with 10 ms hop) across the signal and
computes the magnitude spectrum at each step. The result is a 2-D image
where the x-axis is time, the y-axis is frequency, and pixel intensity
encodes energy.

## 2 · Spectrogram Representation

A raw linear spectrogram allocates equal resolution to every frequency band,
but human hearing is **logarithmic** — we perceive the difference between
200 Hz and 400 Hz as roughly the same as between 2 000 Hz and 4 000 Hz.
The **mel scale** warps the frequency axis to mimic this perceptual spacing.
Applying a bank of triangular filters (typically 80 or 128) to the linear
spectrogram and taking the log of the resulting energies yields the
**log-mel spectrogram**, which is the standard input to modern ASR encoders.

| Representation | X-axis | Y-axis | Best for |
|:---|:---|:---|:---|
| Waveform | Time (samples) | Amplitude | Seeing overall loudness, clipping, silence |
| Spectrogram (STFT) | Time (frames) | Linear frequency (Hz) | Inspecting harmonics, noise bands |
| Mel-spectrogram | Time (frames) | Mel-scale frequency | Model input — emphasizes speech-relevant bands |

Whisper expects an **80-channel log-mel spectrogram** from a 25 ms window
with 10 ms hop on 16 kHz audio, padded or truncated to **30 seconds**
(3 000 frames). Let's build one.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────
!pip install -q transformers torch torchaudio datasets librosa soundfile bitsandbytes accelerate jiwer

import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torchaudio
from IPython.display import Audio, display

print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Synthetic multi-frequency signal ───────────────────────────────
SR = 16_000  # standard speech sampling rate
duration = 2.0  # seconds
t = np.linspace(0, duration, int(SR * duration), endpoint=False)

# Three harmonics + a chirp to simulate speech-like spectral variety
signal = (
    0.5 * np.sin(2 * np.pi * 200 * t)          # fundamental
    + 0.3 * np.sin(2 * np.pi * 600 * t)        # second formant
    + 0.15 * np.sin(2 * np.pi * 1500 * t)      # third formant
    + 0.1 * np.sin(2 * np.pi * (200 + 2000 * t / duration) * t)  # chirp
)
signal = signal / np.max(np.abs(signal))  # normalize

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

# Waveform
axes[0].plot(t[:1600], signal[:1600], linewidth=0.5)
axes[0].set_title("Waveform (first 100 ms)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Linear spectrogram
S = np.abs(librosa.stft(signal, n_fft=1024, hop_length=160))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=SR, hop_length=160, x_axis="time",
                         y_axis="linear", ax=axes[1])
axes[1].set_title("Linear Spectrogram")

# Mel-spectrogram
M = librosa.feature.melspectrogram(y=signal, sr=SR, n_mels=80,
                                   n_fft=1024, hop_length=160)
M_db = librosa.power_to_db(M, ref=np.max)
librosa.display.specshow(M_db, sr=SR, hop_length=160, x_axis="time",
                         y_axis="mel", ax=axes[2])
axes[2].set_title("80-channel Mel-Spectrogram (Whisper input format)")

plt.tight_layout()
plt.show()

## 3 · The Whisper Architecture

OpenAI's **Whisper** (Radford et al., 2022) is an encoder–decoder Transformer
trained on **680,000 hours** of weakly supervised multilingual audio from the
web. Rather than requiring meticulously labelled datasets, Whisper learns from
noisy transcripts, subtitles, and translations — trading label precision for scale.

**Input:** a 30-second log-mel spectrogram (80 × 3 000) is passed through two
convolutional down-sampling layers, then a stack of Transformer encoder
blocks. The decoder attends to the encoder output and autoregressively
produces a sequence of tokens from a multilingual byte-pair vocabulary.

**Special tokens** steer behaviour:

| Token | Purpose |
|:---|:---|
| `<|startoftranscript|>` | Begins generation |
| `<|en|>`, `<|fr|>`, … | Language ID (99 languages) |
| `<|transcribe|>` / `<|translate|>` | Task token — translate always targets English |
| `<|notimestamps|>` | Suppress timestamp prediction |
| `<|0.00|>` … `<|30.00|>` | Segment-level timestamp tokens (in 0.02 s steps) |

This design makes Whisper a **multitask** model: it performs language
detection, transcription, and speech-to-English translation within a single
architecture, controlled entirely by the decoder prompt.

In [ ]:
# ── Load a real audio sample ───────────────────────────────────────
from datasets import load_dataset

ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation",
    trust_remote_code=True,
)
sample = ds[0]
audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]

print(f"Duration : {len(audio_array)/sr:.2f}s  |  SR: {sr} Hz")
print(f"Reference: {sample['text']}")
display(Audio(audio_array, rate=sr))

In [ ]:
# ── Transcribe with Whisper-small ──────────────────────────────────
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    torch_dtype=torch_dtype,
    device=device,
)

result = pipe(audio_array.copy())
print("Transcription:", result["text"])

## 4 · Model Size Trade-offs

Whisper ships in several sizes. Larger models yield lower error rates but
demand more VRAM and run slower. The table below summarizes the landscape:

| Model | Parameters | VRAM (fp16) | Relative Speed | WER (en, LibriSpeech test-clean) |
|:---|---:|---:|---:|---:|
| `whisper-tiny` | 39 M | ~1 GB | 32× | ~7.6 % |
| `whisper-base` | 74 M | ~1 GB | 16× | ~5.0 % |
| `whisper-small` | 244 M | ~2 GB | 6× | ~3.4 % |
| `whisper-medium` | 769 M | ~5 GB | 2× | ~2.9 % |
| `whisper-large-v3` | 1 550 M | ~10 GB | 1× | ~2.0 % |

**Rule of thumb:** start with `whisper-small` for prototyping. It fits
comfortably on a free Colab T4 GPU, transcribes in near real-time, and
delivers solid quality for clean English speech. Move to `large-v3` when
you need multilingual robustness or noisy-environment accuracy, and down
to `tiny` / `base` for latency-critical edge deployment.

In [ ]:
# ── Compare transcription speed across model sizes ─────────────────
# We benchmark tiny and small (loading large models on free Colab is slow)

def benchmark_model(model_id, audio, n_runs=3):
    """Load model, transcribe, return avg time and text."""
    p = pipeline(
        "automatic-speech-recognition",
        model=model_id,
        torch_dtype=torch_dtype,
        device=device,
    )
    # warm-up
    _ = p(audio.copy())
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        out = p(audio.copy())
        times.append(time.time() - t0)
    avg = np.mean(times)
    return avg, out["text"]

results_bench = {}
for mid in ["openai/whisper-tiny", "openai/whisper-small"]:
    avg_t, text = benchmark_model(mid, audio_array)
    results_bench[mid] = {"avg_s": round(avg_t, 3), "text": text}
    print(f"{mid:30s}  avg={avg_t:.3f}s  → {text[:80]}")

# free memory
torch.cuda.empty_cache()

## 5 · Long-Form Transcription

Whisper processes audio in **30-second windows**. For longer recordings the
Hugging Face pipeline offers two chunking strategies:

1. **Sequential chunking** — cut the audio into non-overlapping 30 s blocks.
   Fast but can split words at boundaries.
2. **Sliding window with overlap** — use `stride_length_s=(left, right)` to
   add context before and after each chunk. The pipeline cross-fades the
   overlapping logits, reducing boundary artefacts.

```python
pipe(long_audio, chunk_length_s=30, stride_length_s=(4, 2))
#                 ─────────────     ──────────────────────
#                 chunk size         left / right overlap
```

The stride values mean: prepend 4 s of the previous chunk and append 2 s of
the next chunk to each 30 s window. This gives the decoder enough surrounding
context to avoid hallucinating or repeating words at boundaries.

In [ ]:
# ── Build a ~60 s clip by concatenating samples ────────────────────
long_audio = np.concatenate([np.array(ds[i]["audio"]["array"], dtype=np.float32)
                              for i in range(6)])
print(f"Long audio duration: {len(long_audio)/sr:.1f}s")

# Transcribe with chunking
long_result = pipe(
    long_audio.copy(),
    chunk_length_s=30,
    stride_length_s=(4, 2),
    batch_size=4,
)
print("\n── Long-form transcript ──")
print(long_result["text"][:500])

## 6 · Timestamps and Word-Level Alignment

Whisper can emit **timestamp tokens** that mark when each segment or word
begins and ends. This is invaluable for:

* **Subtitle generation** — SRT / VTT files need start and end times.
* **Karaoke-style highlighting** — word-level timing lets you highlight the
  current word in sync with audio.
* **Video editing** — jump to the exact moment a phrase is spoken.

The Hugging Face pipeline exposes two granularities:

| Argument | Granularity | Output |
|:---|:---|:---|
| `return_timestamps=True` | Segment (~5–10 s) | `{"text": ..., "timestamp": (start, end)}` |
| `return_timestamps="word"` | Word | One entry per word with precise timing |

In [ ]:
# ── Segment-level timestamps ──────────────────────────────────────
seg_result = pipe(audio_array.copy(), return_timestamps=True)
print("── Segment timestamps ──")
for chunk in seg_result["chunks"]:
    s, e = chunk["timestamp"]
    e_str = f"{e:6.2f}" if e is not None else "   end"
    s_str = f"{s:6.2f}" if s is not None else " start"
    print(f"  [{s_str}s → {e_str}s]  {chunk['text'].strip()}")

In [ ]:
# ── Word-level timestamps ─────────────────────────────────────────
word_result = pipe(audio_array.copy(), return_timestamps="word")
print("── Word timestamps (first 15 words) ──")
for w in word_result["chunks"][:15]:
    s, e = w["timestamp"]
    e_str = f"{e:5.2f}" if e is not None else " ???"
    s_str = f"{s:5.2f}" if s is not None else "start"
    print(f"  [{s_str}s → {e_str}s]  {w['text'].strip()}")

## 7 · Language Detection and Multilingual Transcription

Whisper supports **99 languages**. When the decoder prompt contains a
language token (e.g., `<|fr|>`), the model transcribes in that language.
When set to `<|translate|>` instead of `<|transcribe|>`, it produces an
**English translation** of the source speech — effectively performing
speech-to-text translation in a single pass.

If you omit the language token, Whisper spends the first 30 s detecting the
spoken language before transcribing. The pipeline's
`generate_kwargs={"language": "french", "task": "translate"}` provides a
convenient way to control this behaviour.

In [ ]:
# ── Language detection and forced-English transcription ────────────
# The dummy dataset is English, so we demonstrate the API surface.

# Transcribe with explicit language hint
en_result = pipe(
    audio_array.copy(),
    generate_kwargs={"language": "english", "task": "transcribe"},
)
print("Forced English  :", en_result["text"])

# Translation mode (no-op for English input, but shows the API)
tr_result = pipe(
    audio_array.copy(),
    generate_kwargs={"task": "translate"},
)
print("Translate mode  :", tr_result["text"])

## 8 · Quality Metrics — WER and CER

The standard metric for ASR is **Word Error Rate (WER)**:

$$
\text{WER} = \frac{S + D + I}{N}
$$

where *S* = substitutions, *D* = deletions, *I* = insertions, and *N* = total
words in the reference. A WER of 5 % means roughly one error for every 20
words — close to human parity on clean read speech.

**Character Error Rate (CER)** applies the same formula at the character
level. CER is more informative for languages without clear word boundaries
(e.g., Chinese, Japanese) or when you care about partial-word accuracy.

| Scenario | Typical WER |
|:---|---:|
| Clean read speech (LibriSpeech) | 2–5 % |
| Broadcast news | 5–10 % |
| Conversational telephone speech | 10–20 % |
| Meeting with overlapping speakers | 20–40 % |

In [ ]:
# ── WER / CER from scratch ────────────────────────────────────────

def edit_distance(ref_tokens, hyp_tokens):
    """Minimum edit distance via dynamic programming."""
    n, m = len(ref_tokens), len(hyp_tokens)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )
    return dp[n][m]

def compute_wer(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    return edit_distance(ref_words, hyp_words) / max(len(ref_words), 1)

def compute_cer(reference, hypothesis):
    ref_chars = list(reference.lower())
    hyp_chars = list(hypothesis.lower())
    return edit_distance(ref_chars, hyp_chars) / max(len(ref_chars), 1)

# Example
ref = sample["text"]
hyp = result["text"].strip()
print(f"Reference  : {ref}")
print(f"Hypothesis : {hyp}")
print(f"WER (ours) : {compute_wer(ref, hyp):.2%}")
print(f"CER (ours) : {compute_cer(ref, hyp):.2%}")

In [ ]:
# ── Verify with jiwer ─────────────────────────────────────────────
from jiwer import wer as jiwer_wer, cer as jiwer_cer

print(f"WER (jiwer) : {jiwer_wer(ref.lower(), hyp.lower()):.2%}")
print(f"CER (jiwer) : {jiwer_cer(ref.lower(), hyp.lower()):.2%}")

# ── Illustrate S / D / I with a crafted example ───────────────────
ref_ex = "the cat sat on the mat"
hyp_ex = "the bat sat the mat there"  # S=cat→bat, D=on, I=there
print(f"\nCrafted example  WER: {jiwer_wer(ref_ex, hyp_ex):.2%}")
print(f"  ref: {ref_ex}")
print(f"  hyp: {hyp_ex}")
print("  (sub: cat→bat,  del: on,  ins: there  →  3 errors / 6 words = 50%)")

## 9 · Common Failure Modes

Even the largest Whisper model struggles in certain conditions:

| Failure mode | Why it's hard | Mitigation |
|:---|:---|:---|
| **Background noise** | Noise energy masks speech formants | Pre-process with denoising (e.g., DeepFilterNet) |
| **Strong accents** | Training data skews toward standard dialects | Fine-tune on accent-specific data |
| **Overlapping speakers** | Single-channel model assumes one speaker | Use speaker diarization first, then transcribe segments |
| **Domain jargon** | Rare tokens get low probability | Prompt engineering or constrained decoding |
| **Whispered speech** | Very low energy, flat spectrum | Dedicated whispered-speech models |
| **Music with lyrics** | Instruments compete with vocals | Source separation (e.g., Demucs) before ASR |

A robust production pipeline typically chains a **voice activity detector**
(VAD) → **denoiser** → **diarizer** → **ASR** → **punctuation restorer**.

In [ ]:
# ── Noise robustness experiment ────────────────────────────────────

def add_noise(signal, snr_db):
    """Add white Gaussian noise at the specified SNR (dB)."""
    sig_power = np.mean(signal ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.default_rng(42).normal(0, np.sqrt(noise_power), len(signal))
    return (signal + noise).astype(np.float32)

ref_text = sample["text"].lower()
snr_levels = [40, 30, 20, 15, 10, 5, 0]
wer_values = []

for snr in snr_levels:
    noisy = add_noise(audio_array, snr)
    hyp_text = pipe(noisy.copy())["text"].strip().lower()
    w = compute_wer(ref_text, hyp_text)
    wer_values.append(w)
    print(f"  SNR {snr:3d} dB → WER {w:.2%}  | {hyp_text[:60]}")

In [ ]:
# ── Plot WER vs SNR ────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(snr_levels, [w * 100 for w in wer_values], "o-", linewidth=2)
plt.xlabel("Signal-to-Noise Ratio (dB)")
plt.ylabel("Word Error Rate (%)")
plt.title("Whisper-small Robustness to Additive White Noise")
plt.gca().invert_xaxis()  # higher SNR (cleaner) on the left
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10 · Exercises

### Exercise 1 — Noise Robustness Sweep

Extend the noise experiment to include **pink noise** (weighted toward low
frequencies) in addition to white noise. Plot both WER curves on the same
axes and comment on which noise type degrades speech more quickly and why.

### Exercise 2 — Model Size Comparison

Load `whisper-tiny` and `whisper-small`. Transcribe at least 10 samples from
`librispeech_asr_dummy`, compute WER for each model, and report the average.
Is the accuracy gap worth the extra latency on a T4 GPU?

### Exercise 3 — SRT Subtitle Generator

Write a function `generate_srt(audio, pipe) -> str` that returns a valid
SRT subtitle file using word-level timestamps from Whisper. Group words
into subtitle blocks of ≤10 words each.

In [ ]:
# ── Exercise 1 starter ────────────────────────────────────────────

def pink_noise(n, rng=None):
    """Generate pink (1/f) noise of length n."""
    rng = rng or np.random.default_rng(0)
    white = rng.normal(size=n)
    fft = np.fft.rfft(white)
    freqs = np.fft.rfftfreq(n, d=1.0)
    freqs[0] = 1  # avoid division by zero
    fft = fft / np.sqrt(freqs)
    pink = np.fft.irfft(fft, n=n)
    return pink / np.max(np.abs(pink))

# TODO: add pink noise at various SNR levels, transcribe, compute WER,
# and plot white-noise WER vs pink-noise WER on the same chart.
pass

In [ ]:
# ── Exercise 2 starter ────────────────────────────────────────────

def compare_models(dataset, model_ids, n_samples=10):
    """Transcribe n_samples with each model, return dict of avg WER."""
    results = {}
    for mid in model_ids:
        p = pipeline("automatic-speech-recognition", model=mid,
                     torch_dtype=torch_dtype, device=device)
        wers = []
        for i in range(n_samples):
            s = dataset[i]
            audio = np.array(s["audio"]["array"], dtype=np.float32)
            hyp_text = p(audio.copy())["text"].strip()
            wers.append(compute_wer(s["text"].lower(), hyp_text.lower()))
        results[mid] = np.mean(wers)
        print(f"{mid:30s}  avg WER: {results[mid]:.2%}")
        del p
        torch.cuda.empty_cache()
    return results

# TODO: call compare_models(ds, ["openai/whisper-tiny", "openai/whisper-small"])
pass

In [ ]:
# ── Exercise 3 starter ────────────────────────────────────────────

def generate_srt(audio_array, pipe, words_per_block=10):
    """Generate SRT subtitle string from audio using word-level timestamps."""
    result = pipe(audio_array.copy(), return_timestamps="word")
    chunks = result["chunks"]
    srt_blocks = []
    block_idx = 1

    for i in range(0, len(chunks), words_per_block):
        block = chunks[i : i + words_per_block]
        start = block[0]["timestamp"][0] or 0.0
        end = block[-1]["timestamp"][1] or start + 5.0
        text = " ".join(w["text"].strip() for w in block)

        def fmt(t):
            h, rem = divmod(t, 3600)
            m, s = divmod(rem, 60)
            ms = int((s % 1) * 1000)
            return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{ms:03d}"

        srt_blocks.append(f"{block_idx}\n{fmt(start)} --> {fmt(end)}\n{text}\n")
        block_idx += 1

    return "\n".join(srt_blocks)

# TODO: call generate_srt and print the output
# print(generate_srt(audio_array, pipe))
pass

## Key Takeaways

| # | Insight |
|:--|:---|
| 1 | Speech signals are sampled at **16 kHz** and converted to 80-channel **log-mel spectrograms** before entering the model. |
| 2 | Whisper is a multitask encoder–decoder Transformer: it can **transcribe**, **translate**, and **detect language** with a single architecture. |
| 3 | The `transformers` pipeline supports **chunking with overlap** for long-form audio, avoiding boundary artefacts. |
| 4 | **Timestamp tokens** provide segment-level and word-level alignment — essential for subtitle and search applications. |
| 5 | **WER** is the standard metric; Whisper-small achieves ~3–4 % on clean English, degrading gracefully with noise until roughly 5–10 dB SNR. |
| 6 | A production pipeline pairs ASR with upstream components (**VAD, denoiser, diarizer**) and downstream post-processing (**punctuation, casing**). |

## References

1. Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2022). *Robust Speech Recognition via Large-Scale Weak Supervision*. [arXiv:2212.04356](https://arxiv.org/abs/2212.04356)
2. Hugging Face Whisper docs — [huggingface.co/openai/whisper-small](https://huggingface.co/openai/whisper-small)
3. LibriSpeech corpus — [openslr.org/12](https://www.openslr.org/12/)
4. `jiwer` library — [github.com/jitsi/jiwer](https://github.com/jitsi/jiwer)
5. Mel scale — Stevens, S. S., Volkmann, J., & Newman, E. B. (1937). *A Scale for the Measurement of the Psychological Magnitude Pitch*. JASA.

---

⬅️ [Notebook 12 — Multimodal Grounding & Evaluation](12_multimodal_grounding_and_evaluation.ipynb) · [Notebook 14 — Audio Understanding, Classification & Tagging](14_audio_understanding_classification_and_tagging.ipynb) ➡️